## Goal

The goal of this notebook is to scrape data needed for a book recommendation project. The way the project is going to work is it takes an input of Kdramas you have enjoyed and recommend books that are similar to those Kdramas. It's more of a since you liked this drama, you might like this book. It is for those who enjoy reading books and watching Kdramas like me <3.

#### Data Needed
- Kdrama data (title, genre, rating, synopsis, etc.)
- Book data (title, genre, rating, synopsis, etc.)

#### Sources
- Kdrama data: https://mydramalist.com/k-dramas
- Book data: https://www.goodreads.com (albeit obtained from kaggle via https://www.kaggle.com/datasets/ishanrealstate/goodreads-cleaned-dataset)

For the Kdrama data, I couldn't get all I needed in a place so I scraped titles off KdramaList using seleniumand then used the mydramalistAPI to get the rest of the data. As for book data, a more comprehensive dataset was available on kaggle but if you wish to scrape, amazon is a good place to look into. Another good place is using hardcover API.

https://www.amazon.com/s?i=stripbooks&s=relevancerank&Adv-Srch-Books-Submit.x=19&Adv-Srch-Books-Submit.y=9&unfiltered=1&ref=sr_adv_b

https://docs.hardcover.app/api/getting-started/

### Scraping Kdrama Names and Links

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
import random

In [32]:
def scrape_pages(start_page, end_page):
    chrome_path = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"

    service = Service()  # Selenium will try Selenium Manager again
    options = Options()
    options.binary_location = chrome_path

    driver = webdriver.Chrome(service=service, options=options)

    for page in range(start_page, end_page + 1):
        url = f"https://mydramalist.com/search?adv=titles&ty=68,77,86&co=3&rt=4,10&st=3&so=top&page={page}"
        print(f"Scraping page {page}...")
        
        driver.get(url)

        # wait for JS + Cloudflare
        time.sleep(random.uniform(4, 7))

        titles = driver.find_elements(By.CSS_SELECTOR, "h6.text-primary.title a")
        if not titles:
            print(f"No titles found on page {page}")

        with open("title.txt", "a", encoding="utf-8") as f:
            for title in titles:
                text = title.text.strip()
                link = title.get_attribute('href')
                if text and link:
                    f.write(f"{text} | {link}\n")

        time.sleep(random.uniform(2, 5))

    driver.quit()
    print("Batch complete.\n")

In [33]:
total_pages = 250
batch_size = 10

In [ ]:
for start in range(1, total_pages+1, batch_size):
    end = min(start+batch_size-1,total_pages)
    scrape_pages(start,end)
    time.sleep(random.uniform(10,20))

Scraping page 203...
Scraping page 204...
Scraping page 205...
Scraping page 206...
Scraping page 207...
Scraping page 208...
Scraping page 209...
Scraping page 210...
Scraping page 211...
Scraping page 212...
Batch complete.

Scraping page 213...
Scraping page 214...
Scraping page 215...
Scraping page 216...
Scraping page 217...
Scraping page 218...
Scraping page 219...
Scraping page 220...
Scraping page 221...
Scraping page 222...
Batch complete.

Scraping page 223...
Scraping page 224...
Scraping page 225...
Scraping page 226...
Scraping page 227...
Scraping page 228...
Scraping page 229...
Scraping page 230...
Scraping page 231...
Scraping page 232...
Batch complete.

Scraping page 233...
Scraping page 234...
Scraping page 235...
Scraping page 236...
Scraping page 237...
Scraping page 238...
Scraping page 239...
Scraping page 240...
Scraping page 241...
Scraping page 242...
Batch complete.

Scraping page 243...
Scraping page 244...
Scraping page 245...
Scraping page 246...
Scraping

### Scraping full kdrama data from my dramalist API

In [27]:
import pandas as pd
import requests
import json
import time

ModuleNotFoundError: No module named 'requests'

In [ ]:
movies = pd.read_csv('title.txt',sep='|',names=["Title","Links"])

In [ ]:
movies['slug'] = movies['Links'].apply(lambda x: x.split('/')[-1].strip()) #extracting slug from link

In [ ]:
movies

,Title,Links,slug
0,Nana Tour with Seventeen,https://mydramalist.com/757373-youth-over-flo...,757373-youth-over-flowers
1,BTS in the Soop Season 2,https://mydramalist.com/710607-bts-in-the-soo...,710607-bts-in-the-soop-season-2
2,Run BTS! 2022 Special Episode: Fly BTS Fly,https://mydramalist.com/740861-run-bts-2022-s...,740861-run-bts-2022-special-episode-fly-bts-fly
3,Suga: Road to D-Day,https://mydramalist.com/750569-suga-road-to-d...,750569-suga-road-to-d-day
4,When Life Gives You Tangerines,https://mydramalist.com/735043-life,735043-life
...,...,...,...
4995,All About Dong Bang Shin Ki Season 2,https://mydramalist.com/51605-all-about-dong-...,51605-all-about-dong-bang-shin-ki-season-2
4996,Chick High Kick,https://mydramalist.com/696167-chick-high-kick,696167-chick-high-kick
4997,Oh My Girl Miracle Expedition,https://mydramalist.com/27014-oh-my-girl-mira...,27014-oh-my-girl-miracle-expedition
4998,Wanna One Amigo TV Season 2,https://mydramalist.com/59553-wanna-one-amigo-tv,59553-wanna-one-amigo-tv


In [ ]:
movie_slugs = list(movies.slug)

In [ ]:
def write_to_txt(movie_details,output_file,count):
    """helper function: function to write movie details extracted to csv"""
    with open(output_file,'a',encoding='utf-8') as f:
        for md in movie_details:
            f.write(json.dumps(md) + '\n')
        print(f"{count} movie details writen to {output_file}")

In [ ]:
def get_movie_details(slugs, output_file = 'Movie_File.jsonl', batch_size = 50):
    """This function gets movie details from dramalist API and uses write_to_txt function to write to a file"""
    session = requests.Session()
    movie_details = []
    for i, slug in enumerate(slugs, 1):
        link = f"https://my-drama-list-api-ten.vercel.app/api/id/{slug}"
        for attempt in range(3):    
            try:
                response = session.get(link, timeout=10)
                response.raise_for_status()
                movie_detail = response.json()
                movie_details.append(movie_detail)
                break
            except requests.exceptions.RequestException:
                print(f"Attempt {attempt + 1} failed for slug {slug}")
                if attempt == 2:
                    print(f"Failed to get movie details for slug {slug}")
                else:
                    time.sleep(0.5)
        if i % batch_size==0:
            write_to_txt(movie_details, output_file,count=i)
            movie_details = []
        time.sleep(0.2)
    if movie_details:
        write_to_txt(movie_details, output_file,count=i)
    print("All done")
        

In [ ]:
get_movie_details(slugs=movie_slugs[3949:])

Attempt 1 failed for slug 807-the-doll-master
Attempt 2 failed for slug 807-the-doll-master
Attempt 3 failed for slug 807-the-doll-master
Failed to get movie details for slug 807-the-doll-master
Attempt 1 failed for slug 1552-changing-partners
Attempt 2 failed for slug 1552-changing-partners
Attempt 3 failed for slug 1552-changing-partners
Failed to get movie details for slug 1552-changing-partners
50 movie details writen to Movie_File.jsonl
100 movie details writen to Movie_File.jsonl
Attempt 1 failed for slug 2927-the-intimate-lover
Attempt 2 failed for slug 2927-the-intimate-lover
Attempt 3 failed for slug 2927-the-intimate-lover
Failed to get movie details for slug 2927-the-intimate-lover
150 movie details writen to Movie_File.jsonl
Attempt 1 failed for slug 15165-queer-movie-butterfly-the-adult-world
Attempt 2 failed for slug 15165-queer-movie-butterfly-the-adult-world
Attempt 3 failed for slug 15165-queer-movie-butterfly-the-adult-world
Failed to get movie details for slug 15165-

In [ ]:
data = pd.read_json('Movie_File.jsonl',lines = True)

In [ ]:
data.to_csv('raw_kdrama.csv')

### Scraping Book Details

In [ ]:
import pandas as pd
import sys

In [ ]:
book_data = pd.read_parquet('Goodreads Books/books_clean.parquet')

In [ ]:
book_data.to_csv('raw_books_data.csv', index=False)

In [ ]:
book_data[0:100]

,id,name,author,url,genres,summary_clean,pub_year,star_rating,num_ratings,isbn_clean
0,13206035-the-camera-as-historian,The Camera as Historian: Amateur Photographers...,[Elizabeth Edwards],https://www.goodreads.com/book/show/13206035-t...,"[Photography, History, Art History]",In the late nineteenth century and early twent...,2012,3.77,13,9780822350903
1,20575418-soul-of-the-fire,Soul of the Fire,[Eliot Pattison],https://www.goodreads.com/book/show/20575418-s...,"[Mystery, Fiction, Mystery Thriller, Crime, As...",When Shan Tao Yun and his old friend Lokesh ar...,2014,4.43,402,9780312656034
2,4171519-dolphin-societies,Dolphin Societies: Discoveries and Puzzles,[Karen Pryor],https://www.goodreads.com/book/show/4171519-do...,"[Science, Animals, Nonfiction]",Wild dolphins are an elusive subject for behav...,1991,4.00,21,9780520067172
3,1396767.With_Your_Own_Two_Hands,With Your Own Two Hands: Self-Discovery Throug...,[Seymour Bernstein],https://www.goodreads.com/book/show/1396767.Wi...,"[Music, Nonfiction, Art]",This best-seller by the nationally acclaimed p...,1981,4.32,103,9780793557127
4,13093113-flash-from-the-bowery,Flash from the Bowery: Classic American Tattoo...,[Cliff White],https://www.goodreads.com/book/show/13093113-f...,[Art],The only known tattoo art that has survived fr...,2011,4.75,12,9780764339288
...,...,...,...,...,...,...,...,...,...,...
1782249,35062976-la-semilla-del-odio,La semilla del odio: De la invasión de Irak al...,[Mónica G. Prieto],https://www.goodreads.com/book/show/35062976-l...,[Nonfiction],¿En qué momento se desató el caos e imperó el ...,2017,4.38,55,9788499927640
1782250,1665220.Airmail_to_the_Moon,Airmail to the Moon,[Tom Birdseye],https://www.goodreads.com/book/show/1665220.Ai...,"[Picture Books, Childrens, Family, Mystery]",When the tooth that she was saving for the too...,1988,4.04,74,9780823406838
1782251,2573994-ralph-edwards-of-lonesome-lake,Ralph Edwards of Lonesome Lake,[Ed Gould],https://www.goodreads.com/book/show/2573994-ra...,[Biography],"Often called The Crusoe of Lonesome Lake, beca...",1979,4.11,9,9780919654747
1782252,8706822-nunn-s-chess-endings-volume-1,Nunn's Chess Endings Volume 1,[John Nunn],https://www.goodreads.com/book/show/8706822-nu...,[Chess],The definitive work on practical endgame tactics,2010,4.60,15,9781906454210


In [ ]:
for i in range(0,20):
    book